# Лабораторная работа № 2
Тема: Реализация нейросети на PyTorch.
План лабораторной занятия
Цель: освоить базовые принципы работы с фреймворком PyTorch для создания, обучения и
оценки нейронных сетей. Научиться настраивать гиперпараметры и проводить базовые
эксперименты.
Задание:
1. Загрузить датасет MNIST.
2. Построить полносвязную сеть (например, 3 слоя с активацией ReLU).
3. Настроить гиперпараметры:


*   Learning rate (0.001, 0.01, 0.1).
*   Batch size (32, 64, 128).



4. Обучить модель, записывая loss потеря и accuracy.
# Порядок выполнения:
1. Подготовка данных (нормализация, преобразование в тензоры).
2. Определение архитектуры сети.
3. Выбор оптимизатора (Adam, SGD) и функции потерь (CrossEntropy).
4. Обучение с валидацией.
5. Визуализация графиков loss/accuracy.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [2]:
# 1. Подготовка данных

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.Compose([transforms.ToTensor()]))
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.Compose([transforms.ToTensor()]))

lr = 0.01
batch_size = 32

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 137MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 56.0MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 129MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.92MB/s]


In [3]:
# 2. Определение архитектуры сети

model = nn.Sequential(
    nn.Linear(28*28, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

# 3. Выбор оптимизатора и функции потерь

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr)

In [4]:
def accuracy(preds, labels):
    _, predicted = torch.max(preds, 1)
    correct = (predicted == labels).sum().item()
    return correct / labels.size(0)

In [ ]:
# 4. Обучение с валидацией

train_losses = []
test_losses = []
train_accuracy = []
test_accuracy = []
epoches = 10

for epoch in range(epoches):
  model.train()
  train_loss = 0
  train_acc = 0
  for data, labels in train_dataloader:
    data = data.view(data.size(0), -1)
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    train_acc += accuracy(output, labels)

  model.eval()
  test_loss, test_acc = 0, 0
  with torch.no_grad():
    for data, labels in test_dataloader:
      data = data.view(data.size(0), -1)
      output = model(data)
      loss = criterion(output, labels)
      test_loss += loss.item()
      test_acc += accuracy(output, labels)

  train_losses.append(train_loss/len(train_dataloader))
  train_accuracy.append(train_acc/len(train_dataloader))
  test_losses.append(test_loss/len(test_dataloader))
  test_accuracy.append(test_acc/len(test_dataloader))

  print(f"Epoch: {epoch+1}/{epoches}\ttrain_loss: {train_losses[-1]:.4f}\ttrain_accuracy: {train_accuracy[-1]:.4f} \
    \ttest_loss: {test_losses[-1]:.4f}\ttest_accuracy: {test_accuracy[-1]:.4f}")

Epoch: 1/10	train_loss: 1.1455	train_accuracy: 0.7218     	test_loss: 0.4267	test_accuracy: 0.8830
Epoch: 2/10	train_loss: 0.3729	train_accuracy: 0.8945     	test_loss: 0.3158	test_accuracy: 0.9083
Epoch: 3/10	train_loss: 0.3045	train_accuracy: 0.9125     	test_loss: 0.2744	test_accuracy: 0.9192
Epoch: 4/10	train_loss: 0.2650	train_accuracy: 0.9244     	test_loss: 0.2380	test_accuracy: 0.9334
Epoch: 5/10	train_loss: 0.2324	train_accuracy: 0.9332     	test_loss: 0.2106	test_accuracy: 0.9398
Epoch: 6/10	train_loss: 0.2053	train_accuracy: 0.9416     	test_loss: 0.1972	test_accuracy: 0.9425
Epoch: 7/10	train_loss: 0.1831	train_accuracy: 0.9472     	test_loss: 0.1692	test_accuracy: 0.9516


In [ ]:
# Визуализация графиков loss/accuracy.

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="train_losses")
plt.plot(test_losses, label="test_losses")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Losses")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(train_accuracy, label="train_accuracy")
plt.plot(test_accuracy, label="test_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Test Loss")
plt.legend()
plt.show()